# TradeSense · Component 03 · notebook 36: why are some families weak? (diagnostic, CPU, no training)

The weak families of the v2 round are abnormal_volume_burst, trade_flow_imbalance, wash-like (when unseen) and the unseen families. Before building new features, this notebook asks one question per family: **is the family weak because its low-amplitude events cannot be distinguished from normal 1-minute noise (an intrinsic limit), or because the models / features miss something that is visible?**

**Pre-registered reading rule (written before the notebook ran):**
- events of the pooled VAL-3 and CAL-3 sets (VAL-3 scores are out of sample for the base models; pooling gives ~40 events per family and tercile) are split, inside each family, into three amplitude terciles (the generator amplitude `amp`);
- for every family: recall of each final model and of the union (T1c, 2 episodes/day each) per tercile, and the "obvious" fraction = share of events with at least one bar whose largest feature |z| (N0 scaling) exceeds the 99.9th percentile of the same quantity on clean DEV bars;
- **verdict**: if the best model reaches recall ≥ 0.60 in the top amplitude tercile, the family is called *amplitude-limited* (weak only because of low-amplitude events: an honest limit of 1-minute features, reported as such); if it stays below 0.60 even for the top tercile, the family is a *feature/model gap* and it is the target of features v3 (notebook 37), where acceptance is judged on validation per family.
- TEST-3, TEST-B, and every TGNN test day are untouched.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; MODES = ['N0', 'N1']
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
assert (OUT/'declared_final_set_v1.json').exists(), 'run notebook 31 to the end first'
say('protocol B2 intact, sha256 ' + ph[:16])

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f


## 1 · Library (same file as notebooks 28-35)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

library sha256 1b0b46630763c944


## 2 · Data

In [ ]:
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
DF = json.load(open(OUT/'declared_final_set_v1.json')); MODELS = list(DF['models']); FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}
FILES = {'AE-ctx fmax': 'ae_ctx', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}
AOLEN = 31 * 1440; PRIM = B.PRIMARY; L1V = B.valid_mask(129600, 90720, 129600); FEATS = B.FEATS
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
SCR = {}; EV = {}; N0 = {}; GENZ = {}; FZ = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a; a.loc[a.index[:15], 'volume_acceleration'] = 0.0
    g = a[FEATS].reset_index(drop=True); N0[s] = B.fit_n0(g.iloc[15:90720]); GENZ[s] = B.normalise(g, 'N0', N0[s])
    for name in ['VAL', 'CAL']:
        ev = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv'); EV[(s, name)] = ev
        f = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet', columns=FEATS + ['replica'])
        for r, gg in f.groupby('replica'): FZ[(s, name, int(r))] = B.normalise(gg[FEATS].reset_index(drop=True).iloc[:AOLEN] if name == 'CAL' else gg[FEATS].reset_index(drop=True), 'N0', N0[s])
    for model in MODELS:
        z = np.load(OUT/'scores'/f'{FILES[model]}_{s}_{FNORM[model]}.npz'); assert not any('TEST' in k for k in z.files); d = {}
        for k in z.files:
            if k == 'scale' or k.startswith('TRAIN'): continue
            x = z[k]; x = x[:, 0] if x.ndim == 2 else x; d[k] = nanw(x[:AOLEN] if k.startswith('CAL') else x)
        SCR[(model, s)] = d
say('loaded final models ' + str(FNORM) + '; events per set: ' + str({f'{s}|{n}': len(EV[(s, n)]) for s in SYMBOLS for n in ['VAL', 'CAL']}))
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
QC = {}
def alarm(model, s, key, ref, bud=PRIM):
    k = (model, s, bud)
    if k not in QC: QC[k] = B.t1_q(SCR[(model, s)]['GEN_L1'], L1V, bud)
    return SCR[(model, s)][key] > thr_roll(SCR[(model, s)][ref], QC[k])
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(np.nan_to_num(a).astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
CLEAN = np.abs(GENZ[SYMBOLS[0]][B.WARM:]).max(axis=1)

loaded final models {'AE-ctx fmax': 'N1', 'XGBoost v2': 'N0', 'Transformer v2': 'N1'}; events per set: {'BTCUSDT|VAL': 300, 'BTCUSDT|CAL': 200, 'ETHUSDT|VAL': 300, 'ETHUSDT|CAL': 200}


## 3 · Recall by amplitude tercile

In [ ]:
rows = []
for s in SYMBOLS:
    clean_max = np.abs(GENZ[s][90720 + 1440:129600]).max(axis=1); lab_thr = float(np.percentile(clean_max, 99.9))
    for name in ['VAL', 'CAL']:
        ev = EV[(s, name)]
        for r in sorted(ev.replica.unique()):
            e = ev[ev.replica == r].reset_index(drop=True); key = f'{name}_r{int(r)}'; ref = 'GEN_L1' if name == 'VAL' else 'GEN_AO'; Z = FZ[(s, name, int(r))]
            hit = {m: event_hits(alarm(m, s, key, ref), e) for m in MODELS}
            for i, x in enumerate(e.itertuples()):
                zz = np.abs(Z[x.t0:x.t0 + x.dur]); pk = zz.max(axis=1)
                rows.append({'sym': s, 'set': name, 'fam': int(x.family), 'amp': float(x.amp), 'dur': int(x.dur), 'obvious': bool(pk.max() > lab_thr), 'sig': FEATS[int(np.argmax(zz.mean(axis=0)))], **{m: bool(hit[m][i]) for m in MODELS}})
E = pd.DataFrame(rows); E['union'] = E[MODELS].any(axis=1)
E['tercile'] = E.groupby('fam').amp.transform(lambda v: pd.qcut(v, 3, labels=['low', 'mid', 'high']))
say('events pooled (VAL-3 + CAL-3, both symbols): ' + str(len(E)) + '; per family ' + str(E.groupby('fam').size().to_dict()))
OUTROWS = []
for f, g in E.groupby('fam'):
    for t in ['low', 'mid', 'high']:
        h = g[g.tercile == t]; OUTROWS.append({'family': B.FAMS[f], 'amp tercile': t, 'amp range': f'{h.amp.min():.2f}-{h.amp.max():.2f}', 'n': len(h), **{m: float(h[m].mean()) for m in MODELS}, 'union': float(h['union'].mean()), 'obvious share': float(h.obvious.mean())})
T = pd.DataFrame(OUTROWS); say(''); say('--- recall (T1c, 2 episodes/day per model) by amplitude tercile; obvious share = events with a bar beyond the clean 99.9th percentile of max |z| ---'); say(T.round(2).to_string(index=False))

events pooled (VAL-3 + CAL-3, both symbols): 1000; per family {1: 126, 2: 126, 3: 126, 4: 126, 5: 126, 6: 126, 7: 122, 8: 122}

--- recall (T1c, 2 episodes/day per model) by amplitude tercile; obvious share = events with a bar beyond the clean 99.9th percentile of max |z| ---
                   family amp tercile amp range  n  AE-ctx fmax  XGBoost v2  Transformer v2  union  obvious share
       pump_and_dump_like         low 0.51-0.86 42         0.05        0.48            0.50   0.64            0.0
       pump_and_dump_like         mid 0.86-1.34 42         0.29        0.74            0.86   0.88            0.0
       pump_and_dump_like        high 1.34-1.95 42         0.07        0.90            0.98   0.98            0.0
       fake_breakout_like         low 0.50-0.82 42         0.02        0.40            0.57   0.74            0.0
       fake_breakout_like         mid 0.83-1.37 42         0.05        0.74            0.64   0.79            0.0
       fake_breakout_like        high 1

## 4 · Verdict per family and save

In [ ]:
V = []
for f, g in E.groupby('fam'):
    top = g[g.tercile == 'high']; best = max(MODELS + ['union'], key=lambda m: top[m].mean()); br = float(top[best].mean()); allbest = float(g['union'].mean())
    verdict = 'amplitude-limited (weak mainly at low amplitude)' if br >= 0.60 else 'FEATURE/MODEL GAP (weak even at high amplitude) -> features v3'
    dur = g.groupby(pd.qcut(g.dur, 3, labels=['short', 'mid', 'long']))['union'].mean().round(2).to_dict()
    V.append({'family': B.FAMS[f], 'best model in top tercile': best, 'recall there': round(br, 2), 'recall all terciles (union)': round(allbest, 2), 'union recall by duration': dur, 'signature feature (most common)': g.sig.mode().iloc[0], 'verdict': verdict})
VT = pd.DataFrame(V); say(''); say('--- verdict per family (pre-registered rule: top-tercile recall >= 0.60 -> amplitude-limited) ---'); say(VT.to_string(index=False))
gap = [r['family'] for r in V if r['verdict'].startswith('FEATURE')]; say(''); say('families that need features v3: ' + (', '.join(gap) if gap else 'none (all are amplitude-limited)'))
T.to_csv(OUT/'diag36_recall_by_amplitude.csv', index=False); VT.to_csv(OUT/'diag36_verdict.csv', index=False); E.to_csv(OUT/'diag36_event_table.csv', index=False)
block = '\n'.join(LINES); (OUT/'b2_step7_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step7_share_block.txt, or copy below) =====\n' + block[-12000:])


--- verdict per family (pre-registered rule: top-tercile recall >= 0.60 -> amplitude-limited) ---
                   family best model in top tercile  recall there  recall all terciles (union)                   union recall by duration signature feature (most common)                                                        verdict
       pump_and_dump_like            Transformer v2          0.98                         0.83  {'short': 0.73, 'mid': 0.8, 'long': 0.98}                     log_returns               amplitude-limited (weak mainly at low amplitude)
       fake_breakout_like                     union          0.98                         0.83 {'short': 0.74, 'mid': 0.96, 'long': 0.88}                     log_returns               amplitude-limited (weak mainly at low amplitude)
    abnormal_volume_burst                     union          0.38                         0.23 {'short': 0.15, 'mid': 0.24, 'long': 0.34}             volume_acceleration FEATURE/MODEL GAP (weak even at 